In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E07_recomendacion"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión EPE E7 · Sistemas de recomendación y proyecto integrador

**Curso "Herramientas de Ciencias de Datos" · Modalidad EPE · UPC · Facultad de Negocios**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E07_recomendacion/notebook/EPE_S7_recomendacion_proyecto.ipynb)

> Enfoque EPE: se prioriza la **intuición y la decisión de negocio** sobre el
> formalismo matemático. Se construye un recomendador por **similitud** (item-based)
> y por **factorización (SVD)**, se genera el **top-N por usuario**, se **evalúa de
> forma sencilla** (precisión@10 / NDCG@10) y se lleva a un caso de **negocio**
> (feedback implícito). El **deep learning de recomendación queda fuera de alcance**
> (solo se menciona). **Última sesión del curso: cierra el proyecto integrador.**

## Objetivos de aprendizaje
Al terminar la sesión, el participante es capaz de:
1. Explicar el **filtrado colaborativo** y la **matriz usuario-ítem** (Amazon, Netflix).
2. Construir un recomendador por **similitud entre productos** (item-based) y por **factorización (SVD)**, e interpretar sus resultados sin fórmulas.
3. Generar el **top-N por usuario** y **evaluar** su calidad de forma básica (**precisión@10 / NDCG@10** frente a baselines).
4. Aplicar un recomendador de **negocio** con **feedback implícito** (compras) y traducirlo a **cross/up-selling**.
5. **Cerrar el proyecto integrador:** integrar E1–E7 en un caso CRISP-DM y preparar el mensaje ejecutivo.

## Mapa de la sesión
| # | Bloque | Datos |
|---|---|---|
| a | Cómo funcionan las recomendaciones: filtrado colaborativo y la matriz usuario-ítem | MovieLens 100k |
| b | Recomendar por **similitud** (item-based) y por **factorización** (SVD) + top-N | MovieLens 100k |
| c | Recomendación **basada en contenido**, **híbridos** y **cold start** | MovieLens 100k |
| d | **Evaluar** de forma sencilla: precisión@10 / NDCG@10 (partición temporal) | MovieLens 100k |
| e | Recomendador de **negocio**: feedback implícito | Online Retail II |
| f | Exportación a Excel + figuras de resultados | — |
| g | Cierre del **proyecto integrador** (CRISP-DM end-to-end) | — |

**Materiales hermanos:** guía de laboratorio `laboratorio/GUIA_LABORATORIO_E07.docx`,
plantillas `plantillas/guia_proyecto_integrador_epe.docx`, `plantillas/repositorio_plantilla_epe.docx`
y `plantillas/plantilla_sustentacion_epe.docx`, ejercicios `evaluacion/drills.docx`,
entregable `evaluacion/entregable.docx`, fuentes las fuentes de actualidad de la sesión,
diccionario `data/diccionario_datos.md`.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q implicit scikit-surprise

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "numpy": "2.5.1",
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


In [ ]:
# Librerías de la sesión
import os, sys
import warnings; warnings.simplefilter("ignore")
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import surprise
from surprise import Dataset, Reader, SVD, KNNBaseline, KNNBasic, NormalPredictor, accuracy
from surprise.model_selection import train_test_split

print("surprise", surprise.__version__, "| pandas", pd.__version__,
      "|", "Colab" if "google.colab" in sys.modules else "entorno local")

# Estética (paleta de marca UPC: rojo + neutros)
UPC_RED, UPC_MAROON, INK, GRIS = "#E4002B", "#9B1B30", "#2D2D2D", "#6E6E6E"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
                     "axes.titleweight": "bold", "axes.grid": True, "grid.alpha": 0.3,
                     "figure.autolayout": True})

In [ ]:
# === DATOS DE LA SESION EMBEBIDOS (para Google Colab) ======================
# En Colab no existe la carpeta data/ del curso. Los CSV de la sesion viajan
# comprimidos DENTRO de este cuaderno y se escriben en el directorio de trabajo
# antes de cargarse: son los MISMOS bytes que en local (mismo SHA256), de modo
# que las cifras de la sesion no cambian. En local esta celda no hace nada.
import base64, gzip, os, sys

_EMBEBIDOS = {
    "ml100k_ratings.csv":  # 2,079,207 B  sha256=ff8ecf3b80823ffa...
        "...",
    "ml100k_movies.csv":  # 135,229 B  sha256=f6f6427695d2f425...
        "...",
    "online_retail_reco_muestra.csv":  # 794,963 B  sha256=f0ef2267cba1cf7e...
        "...",
}

def _escribir_embebidos(destinos=None):
    """Vuelca los CSV embebidos al directorio actual y a ../data: los cuadernos
    del curso leen de uno o de otro segun como calculen su carpeta de datos."""
    if destinos is None:
        destinos = [os.getcwd(), os.path.join(os.getcwd(), "data"),
                    os.path.join(os.path.dirname(os.getcwd()), "data")]
    escritos = []
    for _dest in destinos:
        try:
            os.makedirs(_dest, exist_ok=True)
        except OSError:
            continue
        for _nombre, _b64 in _EMBEBIDOS.items():
            _ruta = os.path.join(_dest, _nombre)
            if not os.path.exists(_ruta):
                with open(_ruta, "wb") as _fh:
                    _fh.write(gzip.decompress(base64.b64decode(_b64)))
                escritos.append(_nombre)
    return sorted(set(escritos))

if "google.colab" in sys.modules:
    _e = _escribir_embebidos()
    print("Datos de la sesion listos en Colab:", ", ".join(_e) if _e else "ya estaban")


In [ ]:
# Rutas robustas (funcionan con nbconvert local y en Colab) y carga de datos.
import io, hashlib
def _base_dir():
    if "google.colab" in sys.modules:
        return os.getcwd()
    d = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(d, "la ficha de la sesiónmd")):
            return d
        d = os.path.dirname(d)
    d = os.getcwd()
    return os.path.dirname(d) if os.path.basename(d).lower() == "notebook" else d

BASE = _base_dir()
DATA = os.getcwd() if "google.colab" in sys.modules else os.path.join(BASE, "data")
RES = os.path.join(BASE, "resultados"); FIG = os.path.join(BASE, "figuras")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True); os.makedirs(DATA, exist_ok=True)
XLSX = os.path.join(RES, "E07_resultados.xlsx")

# En local, data/descargar_datos.py ya pobló la carpeta. Como respaldo portable
# (Colab), se leen los CSV del repo del curso.
RAW = ("https://raw.githubusercontent.com/jonatanfigueroagil-creator/"
       "Herramientas-de-Ciencias-de-Datos/master/Sesiones_EPE/E07_recomendacion/data/")
# SHA256 esperado de cada CSV: verifica INTEGRIDAD (evita deriva silenciosa) y, si el
# archivo local falta o no cuadra, dispara el fallback de descarga del repo (RAW).
SHA256 = {
    "ml100k_ratings.csv":             "ff8ecf3b80823ffa57f060334a671fde0df2a4c9d3760d1101e5e3ee19c1810c",
    "ml100k_movies.csv":              "f6f6427695d2f425e9ffa165f9b1057f2f5f5da9ae04e763a1af1afcab10ee14",
    "online_retail_reco_muestra.csv": "f0ef2267cba1cf7e665aee827100b821cdb8a37478a4de501b2fcc2d3f3bc632",
}
def _cargar(nombre):
    """Lee el CSV local; si falta o su checksum no coincide, baja del repo (RAW) y verifica."""
    p = os.path.join(DATA, nombre); datos = None
    if os.path.exists(p):
        with open(p, "rb") as fh:
            datos = fh.read()
        if nombre in SHA256 and hashlib.sha256(datos).hexdigest() != SHA256[nombre]:
            print(f"  ! {nombre}: checksum local distinto del esperado; se baja del repo.")
            datos = None
    if datos is None:
        import urllib.request
        with urllib.request.urlopen(RAW + nombre) as r:
            datos = r.read()
        if nombre in SHA256 and hashlib.sha256(datos).hexdigest() != SHA256[nombre]:
            print(f"  ⚠️  {nombre}: checksum remoto tampoco coincide (¿repo actualizado?).")
    return pd.read_csv(io.BytesIO(datos))

ratings = _cargar("ml100k_ratings.csv")   # user_id, item_id, rating (1-5), timestamp
movies  = _cargar("ml100k_movies.csv")    # movie_id, title + 19 flags de género
print("MovieLens ratings:", ratings.shape, "| movies:", movies.shape)

---
## a) Cómo funcionan las recomendaciones: el filtrado colaborativo

Cuando Amazon dice *«los clientes que compraron esto también compraron…»* o Netflix
arma el carrusel *«para ti»* de cada usuario, detrás hay un **sistema de recomendación**. La mayoría
parte de una **matriz usuario-ítem**: una tabla extensa de **usuarios × ítems** donde
cada celda guarda la **calificación** (feedback explícito, 1-5 estrellas) o la
**interacción** (feedback implícito, compró/vio). La inmensa mayoría de las celdas
está **vacía**: nadie califica todo. **Recomendar es estimar las celdas vacías** y
proponer las de mayor valor previsto.

El **filtrado colaborativo** hace eso mirando el **comportamiento de muchos usuarios**,
sin describir el ítem: «personas con gustos parecidos disfrutaron esto». Su enemigo es
la **dispersión (sparsity)**: dos usuarios casi nunca han calificado los mismos ítems.

**❓ Qué se quiere averiguar.** ¿Qué fracción de la matriz usuario-ítem está realmente llena, es decir, de cuánta información dispone el recomendador antes de recomendar nada?

- **Qué decide:** si el tablero estuviera casi lleno, bastaría con ordenar lo que cada cliente ya calificó. Si está casi vacío, recomendar es **estimar casillas en blanco**, y ese vacío es lo que obliga a todas las técnicas del resto de la sesión.
- **Antes de mirar el resultado:** MovieLens 100k son 943 usuarios por 1.682 películas, más de un millón y medio de celdas posibles para 100.000 calificaciones. Si el llenado resultara por encima del **50 %**, el problema sería de ordenar y no de estimar. Si resulta del orden del **6 %**, entonces ese **~94 % de dispersión** no es un defecto que haya que arreglar: es la condición del problema, y de ella nacen el vecindario, la factorización y el arranque en frío.

🔎 **Qué hace el código.** La siguiente celda arma una **matriz usuario-ítem 4×5
ilustrativa** (con huecos) para ver la dispersión por inspección visual; la de después mide la **matriz
real** de MovieLens (943×1.682) y su porcentaje de vacío.

In [ ]:
# Mini-demo: una matriz usuario-ítem 4x5 con huecos ('.' = celda vacía)
demo = pd.DataFrame(
    [[5, np.nan, 4, np.nan, 1],
     [np.nan, 4, np.nan, 2, 1],
     [4, np.nan, 5, np.nan, np.nan],
     [np.nan, 5, np.nan, np.nan, 2]],
    index=["Ana", "Beto", "Cid", "Dina"],
    columns=["Accion1", "Drama1", "Accion2", "Comedia1", "Comedia2"])
print(demo.to_string(na_rep=" ."))
llenas, celdas = int(demo.notna().sum().sum()), demo.size
print(f"\nCeldas: {celdas}  llenas: {llenas}  vacías: {celdas-llenas}")
print(f"Densidad = {llenas/celdas:.0%}  ->  dispersión = {1-llenas/celdas:.0%}")
print("Recomendar = ESTIMAR las celdas vacías (.) y proponer las de mayor valor.")

📖 **Lectura de negocio.** Esta matriz es la representación única de la que parte todo
recomendador. Su tamaño y su **vacío** definen el problema: catálogos de millones de
ítems y clientes, en los que cada cliente ha interactuado con una fracción mínima. Se verá la
matriz **real** de MovieLens y su dispersión.

💡 **Intuición.** La matriz usuario-ítem es el **tablero** del que parten Amazon y
Netflix: las filas son clientes, las columnas productos y casi todo el tablero está
**en blanco**. Recomendar es **estimar las casillas en blanco** de mayor valor para
cada cliente.

In [ ]:
# MovieLens 100k real: tamaño, media y dispersión
n = len(ratings); nu = ratings["user_id"].nunique(); ni = ratings["item_id"].nunique()
mu = float(ratings["rating"].mean()); sigma = float(ratings["rating"].std())
densidad = n / (nu * ni)
print(f"calificaciones = {n:,}")
print(f"usuarios = {nu}   películas = {ni}")
print(f"media global de las notas = {mu:.2f}  (escala 1-5)")
print(f"densidad = {densidad:.2%}   ->   dispersión (sparsity) = {1-densidad:.1%}")
ratings.head()

📖 **Lectura de negocio.** MovieLens 100k tiene **100.000 calificaciones** de **943
usuarios** sobre **1.682 películas**: solo alrededor del **6 %** de la matriz está
lleno (**~94 % de dispersión**). Ese vacío es la razón de ser de toda la sesión. Para
la factorización de notas explícitas **no se rellena con ceros**: un 0 no es una nota
en escala 1-5 (trataría «no visto» como «lo odió»); el modelo aprende **solo de lo
observado**.

⚠️ **Alerta (dispersión y ceros).** El **93,7 % vacío** no es un defecto a «arreglar»:
es la **condición del problema** (nadie califica todo), y de él nacen la dispersión y el
arranque en frío. Rellenar los huecos con 0 sesga el modelo hacia «no le gusta».

---
## b) Recomendar por similitud (item-based) y por factorización (SVD)

Dos formas clásicas de rellenar el vacío, **sin fórmulas**:

- **Por similitud entre productos (item-based).** Se mide qué películas son
  «parecidas» según cómo las calificó la gente y se recomienda lo similar a lo que ya
  gustó. Es el clásico «quien compró esto también compró…». Escala bien (los ítems
  cambian menos que los usuarios) y la similitud se **precalcula**.
- **Por factorización de matrices (SVD).** Resume la matriz completa en pocos
  **factores latentes** — el «ADN de gustos» de cada usuario y cada película (p. ej.
  «acción vs. drama», «para niños»). Es la técnica que ganó el **Netflix Prize**.

> **Intuición, no álgebra.** El «SVD» de la librería `surprise` es la **factorización
> de matrices** de Koren (2009), aprendida solo sobre las notas observadas — **no** la
> descomposición en valores singulares del álgebra lineal. El nombre es histórico; en
> EPE basta la intuición de «factores latentes = resumen de gustos».

Todo modelo se mide **contra un suelo**: el **baseline de media** (predice la nota
media a todos) y el **azar**. El criterio es el **RMSE** (error de predicción de la nota,
en la escala 1-5): **menor = mejor**, y solo tiene sentido **comparado con el suelo**.

**❓ Qué se quiere averiguar.** ¿Cuánto mejor acierta un recomendador que el suelo —predecir la nota media a todo el mundo—, y cuál de los dos vecindarios conviene construir, el de productos o el de usuarios?

- **Qué decide:** si la ventaja sobre la media fuera pequeña, no hay caso de negocio: se publica la media y se ahorra el sistema. Y entre item-based y user-based no se elige solo por error: la similitud entre **productos** se precalcula de madrugada, mientras que la de **usuarios** caduca en cuanto ese usuario actúa.
- **Antes de mirar el resultado:** el RMSE está en la escala 1-5 y **menor es mejor**, pero no se lee aislado. Si los modelos quedaran cerca del baseline de media (**~1,13**) o del azar (**~1,52**), no habría nada que comprar. Si bajan a **~0,93**, la ventaja sobre el suelo es real. Y conviene anticipar la dirección de Sarwar antes de ver la tabla: el **item-based** debería quedar por delante del **user-based**; lo que se replica es ese **orden**, nunca el decimal del paper.

🔎 **Qué hace el código.** La siguiente celda entrena cuatro modelos sobre un único
**split 80/20** —**SVD** (factorización), **item-based** y **user-based** (vecindario)—
y mide el **RMSE** de cada uno frente a dos suelos (**media global** y **azar**); la
tabla se ordena de menor a mayor error.

### Qué preguntaban los dos papers de esta sesión

**💡 Antes de ejecutar el código.** Los dos artículos seminales de esta sesión **no** son dos
intentos sucesivos de recomendar mejor: son dos respuestas a preguntas de ingeniería distintas,
y de ahí salen todas sus decisiones de método. **Sarwar (2001) preguntó CUÁNDO se puede calcular
la recomendación; Hu, Koren & Volinsky (2008) preguntaron QUÉ se puede modelar cuando el usuario
nunca califica nada.** *(Desarrollo completo, con las citas de los originales:
la ficha de la sesión de réplica del paper)*

**Sarwar: la pregunta era el tiempo de respuesta, no el error.** El artículo abre con la
escalabilidad: «*El primer desafío es mejorar la escalabilidad de los algoritmos de filtrado
colaborativo. Estos algoritmos pueden buscar entre decenas de miles de vecinos potenciales en
tiempo real, pero lo que exigen los sistemas modernos es buscar entre decenas de millones*»
(Sección 1). Lo que estaba en juego no era un decimal de error: era si un sitio de comercio
electrónico podía **poner una lista personalizada en la página mientras el cliente la mira**. De
ahí la decisión de método que se prueba en la celda siguiente: la similitud entre **ítems** y no
entre usuarios, porque «*las relaciones entre ítems son relativamente estáticas*» y se precalculan
**fuera de línea**, mientras que el vecindario de un usuario caduca en cuanto ese usuario actúa. La
escalabilidad no se logró con más capacidad de cómputo, sino al **mover el cómputo de la hora punta a la
madrugada**.

**Hu: la pregunta era anterior, qué medir cuando nadie califica.** Su punto de partida es una
carencia declarada en el resumen: «*carecemos de evidencia sustancial sobre qué productos les
disgustan*» (Resumen, p. 263) — la situación normal de una empresa, donde hay compras, clics y
visitas, pero ninguna nota del 1 al 5. Ese dato no admite el trasplante del modelo explícito,
porque «*el valor numérico del feedback explícito indica preferencia; el del feedback implícito
indica confianza*» (Sección 1, p. 264): la película preferida se ve una sola vez, mientras que la
serie que apenas gusta se ve todas las semanas. De ahí las dos piezas del bloque *e*: **preferencia**
binaria y **confianza** `c = 1 + α·cantidad` sobre **todos** los pares cliente-producto, y
**mínimos cuadrados alternados** (ALS) como única forma de mirar la matriz entera sin submuestrear.

> **⚠️ Qué se toma de cada uno.** De Sarwar se reproduce la **dirección**, no su número: el paper
> reportó MAE con particiones propias y coseno ajustado, y aquí se compara el **RMSE** de un split
> 80/20; la lectura legítima es el **orden entre esquemas**, nunca la coincidencia decimal. De Hu
> **no se replica ninguna cifra** —el dataset de televisión del artículo nunca se publicó—: el caso
> de negocio del bloque *e* **ejercita** su modelo de preferencia y confianza.

In [ ]:
# Un solo split 80/20 (más simple que la validación cruzada; suficiente para la intuición EPE)
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[["user_id", "item_id", "rating"]], reader)
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

# 1) SVD (factorización) — factores latentes
#    n_factors=100 y n_epochs=20 son los valores de referencia de surprise/Koren (2009):
#    ~100 factores resumen los gustos sin sobre-ajustar y 20 pasadas bastan para converger.
svd = SVD(n_factors=100, n_epochs=20, random_state=42).fit(trainset)
pred_svd = svd.test(testset)
rmse_svd = accuracy.rmse(pred_svd, verbose=False)
mae_svd = accuracy.mae(pred_svd, verbose=False)

# 2) Similitud item-based (KNNBaseline, similitud entre ítems)
knn_item = KNNBaseline(sim_options={"name": "pearson_baseline", "user_based": False}).fit(trainset)
rmse_item = accuracy.rmse(knn_item.test(testset), verbose=False)

# 3) Similitud user-based (KNNBasic, similitud entre usuarios) — para contrastar
knn_user = KNNBasic(sim_options={"user_based": True}).fit(trainset)
rmse_user = accuracy.rmse(knn_user.test(testset), verbose=False)

# 4) Baselines: media global y azar
mu_train = trainset.global_mean
rmse_media = float(np.sqrt(np.mean([(true - mu_train) ** 2 for (_, _, true) in testset])))
# NormalPredictor muestrea de una normal usando el RNG GLOBAL de numpy: se fija la
# semilla 42 justo antes para que el RMSE del azar sea REPRODUCIBLE (semilla 42, Sección 5).
np.random.seed(42)
azar = NormalPredictor().fit(trainset)
rmse_azar = accuracy.rmse(azar.test(testset), verbose=False)

tabla_rmse = pd.DataFrame([
    ("SVD (factorización, factores latentes)", "factorización", rmse_svd),
    ("Similitud item-based (productos)",       "vecindario",    rmse_item),
    ("Similitud user-based (usuarios)",        "vecindario",    rmse_user),
    ("Media global (baseline)",                "suelo",         rmse_media),
    ("Azar (baseline)",                        "suelo",         rmse_azar),
], columns=["modelo", "tipo", "RMSE"]).sort_values("RMSE").reset_index(drop=True)
print(tabla_rmse.round(3).to_string(index=False))
print(f"\nSVD mejora {1-rmse_svd/rmse_media:+.0%} frente al baseline de media.")

📖 **Lectura de negocio.** Ordenados por error (menor = mejor): la **factorización (SVD)**
y la **similitud item-based** lideran, con un RMSE de **alrededor de 0,93** frente a
**~1,13** del baseline de media y **~1,52** del azar. Dos mensajes:

- La **similitud entre productos** (item-based) supera con holgura a la **similitud entre
  usuarios** (RMSE ~0,98): además de más preciso, tiene **menor costo de mantenimiento**. Es el
  motor del «quien compró esto también compró…».
- La **factorización** iguala o supera al vecindario y resume la matriz completa en pocos
  factores: los **factores latentes** generalizan a pares nunca vistos.

Un RMSE «0,93» **no** se lee aislado: solo dice algo **frente al suelo** (media, azar).

💡 **Intuición (factores latentes = ADN de gustos).** La factorización resume a cada
usuario y a cada película en pocos **factores latentes** —dimensiones implícitas como
«acción vs. drama» o «para niños»—: son su **ADN** de gustos. La nota prevista es la
afinidad entre el ADN del cliente y el del producto. Es la idea que ganó el **Netflix
Prize**.

⚠️ **Alerta (qué mide el RMSE).** El RMSE mide el **error de la nota**, **no** la calidad
de la **lista** que ve el cliente. Un RMSE bajo no garantiza un buen carrusel: el orden
de la lista se evalúa aparte, en el bloque *d*.

**❓ Qué se quiere averiguar.** ¿Qué cinco películas concretas se le ponen a un cliente en el carrusel «para ti»?

- **Qué decide:** esta lista es el producto final, la que viaja al email o a la página de inicio. Todo lo anterior —la matriz, los factores latentes, el RMSE— existe para llenar esas cinco filas.
- **Antes de mirar el resultado:** una lista utilizable tiene que cumplir dos condiciones antes de discutir si acierta. Que **ninguna** de las cinco sea una película que el usuario ya vio, porque recomendar lo ya consumido desperdicia el espacio más valioso de la pantalla; y que las notas estimadas caigan **dentro de la escala 1-5**. Si aparecieran títulos ya vistos, el filtro de «no vistas» falló y la lista no sale a producción por buena que sea su métrica.

🔎 **Qué hace el código.** La siguiente celda genera el **producto final** del
recomendador: el **top-5** de un usuario. Reentrena la SVD con **todos** los datos,
puntúa las películas que ese usuario **no vio** y devuelve las 5 de mayor nota estimada.

In [ ]:
# Top-N por usuario (el producto final del recomendador): las 5 mejores no vistas, con SVD
titulo = movies.set_index("movie_id")["title"].to_dict()
svd_full = SVD(n_factors=100, n_epochs=20, random_state=42).fit(data.build_full_trainset())

def top_n_svd(uid, N=5):
    vistas = set(ratings.loc[ratings.user_id == uid, "item_id"])
    cand = [i for i in ratings.item_id.unique() if i not in vistas]
    est = [(i, svd_full.predict(uid, i).est) for i in cand]
    return sorted(est, key=lambda x: x[1], reverse=True)[:N]

U = 1
print(f"Top-5 recomendaciones (SVD) para el usuario {U} — películas que aún no vio:")
for i, e in top_n_svd(U, 5):
    print(f"  nota estimada {e:.2f}  |  [{i}] {titulo.get(int(i), '?')[:48]}")

📖 **Lectura de negocio.** El **top-N por usuario** es el **producto final**: el carrusel
«para ti». Se ordena por nota estimada y se **excluye lo ya visto**. Es la materia
prima de la personalización — el email, la página de inicio, la ficha de producto — y
lo que efectivamente ve el cliente (su calidad se mide con métricas top-k, bloque *d*, no
con el RMSE).

💡 **Intuición.** Al cliente no le importa la nota prevista (un «4,7»): le importa **qué
cinco títulos** aparecen en su carrusel. Por eso el entregable es una **lista**, no un
número.

---
## c) Recomendación basada en contenido, híbridos y arranque en frío

El **filtrado basado en contenido** no usa a otros usuarios: recomienda ítems
**parecidos en atributos** a los que gustaron (aquí el atributo es el **género**). Se
arma un **perfil de gustos** del usuario y se puntúan las películas no vistas por su
parecido con ese perfil.

**❓ Qué se quiere averiguar.** ¿Qué se le recomienda a quien acaba de registrarse, o a un producto lanzado esta mañana, cuando no hay conducta que observar?

- **Qué decide:** la pantalla del primer día. Un cliente nuevo sin recomendación útil es un cliente que no vuelve, y el colaborativo, sin historial, carece de información de partida.
- **Antes de mirar el resultado:** el content-based solo necesita los **atributos** del ítem (aquí, 19 géneros), no el historial de otros usuarios. Si su top-5 resultara **igual** al del colaborativo, la sesión no habría ganado nada con él. Si resulta **distinto**, cada uno cubre lo que al otro le falta: el contenido es transparente pero se sobre-especializa —más de lo mismo—, y el colaborativo descubre afinidades no obvias pero exige historial. Esa diferencia es todo el argumento del **híbrido**.

🔎 **Qué hace el código.** La siguiente celda construye un **perfil de género** del
usuario (a partir de lo que puntuó alto) y ordena las películas no vistas por su
**parecido de coseno** con ese perfil; luego se contrasta ese top-5 de **contenido** con
el **colaborativo** de arriba para ver que descubren títulos distintos.

In [ ]:
# Content-based con los géneros de MovieLens (19 flags 0/1)
GENRES = ["unknown","Action","Adventure","Animation","Children","Comedy","Crime",
          "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
          "Romance","Sci-Fi","Thriller","War","Western"]
G = movies.set_index("movie_id")[GENRES].astype(float)
Gn = G.div(np.sqrt((G**2).sum(axis=1)).replace(0, 1), axis=0)   # normalizado (coseno)

def top_n_contenido(uid, N=5, umbral=4):
    gustadas = ratings[(ratings.user_id == uid) & (ratings.rating >= umbral)]
    if gustadas.empty:
        return []
    perfil = G.loc[G.index.intersection(gustadas.item_id)].sum()
    perfil = perfil / (np.sqrt((perfil**2).sum()) or 1)
    puntaje = Gn.dot(perfil.values)
    puntaje = puntaje.drop(index=[i for i in ratings.loc[ratings.user_id == uid, "item_id"]
                                  if i in puntaje.index], errors="ignore")
    top = puntaje.sort_values(ascending=False).head(N)
    return [(int(i), titulo.get(int(i), "?"), float(s)) for i, s in top.items()]

perfil = G.loc[G.index.intersection(ratings[(ratings.user_id==U)&(ratings.rating>=4)].item_id)].sum()
print(f"Géneros preferidos del usuario {U} (perfil de contenido):")
print("  " + ", ".join(f"{g}={int(w)}" for g, w in perfil.sort_values(ascending=False).head(5).items()))
print(f"\nTop-5 CONTENT-BASED (por género) para el usuario {U}:")
for i, t, s in top_n_contenido(U, 5):
    print(f"  parecido {s:.2f}  |  [{i}] {t[:48]}")
print("\n(Compárese con el top-5 COLABORATIVO/SVD de arriba: descubren cosas distintas.)")

📖 **Lectura de negocio.** El **content-based** es transparente («te lo recomiendo porque
es del género que te gusta») y funciona **sin historial de otros usuarios**, pero tiende
a la **sobre-especialización** (solo más de lo mismo). El **colaborativo** descubre
afinidades no obvias («quien vio A también disfrutó B»), aunque necesita historial.
Ninguno resulta superior en todos los casos: la respuesta de producción es **híbrida**.

- **Sistemas híbridos:** combinan colaborativo + contenido para cubrir las debilidades
  de cada uno.
- **Arranque en frío (cold start):** un **usuario o ítem nuevo** no tiene historial, y
  el colaborativo devuelve, en la práctica, la **media/popularidad** — no es un fallo,
  es **falta de datos**. La salida es **híbrida**: usar contenido, popularidad o
  preguntas de *onboarding* hasta acumular interacciones.

⚠️ **Alerta (cold start).** No se puede recomendar por conducta a **quien no tiene
conducta**: el producto recién lanzado o el cliente que acaba de registrarse no tienen
fila/columna útil. No es un error del algoritmo, es **falta de datos**; se cubre con
**contenido, popularidad u onboarding**.

> **Fuera de alcance (solo mención):** los recomendadores de *deep learning* (redes
> neuronales, *embeddings* profundos, modelos secuenciales) llevan esta misma idea más
> lejos, pero no se cubren en este curso.

---
## d) Evaluar de forma sencilla: precisión@10 y NDCG@10

El RMSE mide **predicción de nota**, no **calidad de la lista**. El usuario ve una
**lista corta** (top-10), así que se evalúa con métricas **top-k** (intuición):

- **Precisión@10:** de las 10 propuestas, ¿cuántas resultan realmente relevantes? (relevante =
  nota real ≥ 3,5). Mide la **calidad del escaparate**.
- **NDCG@10:** ¿queda lo relevante **arriba**, donde están los clics? Premia el **orden**
  (acertar en el puesto 1 vale más que en el 10); va de 0 a 1.

**Partición temporal (no aleatoria):** se ordena por fecha y se entrena con el **pasado**
para evaluar en el **futuro** — el escenario real. Barajar dejaría «ver el futuro» al
modelo (*data leakage*) e inflaría el resultado. Se compara la **SVD** con dos
baselines: **popularidad** (recomendar los éxitos) y **azar**.

**❓ Qué se quiere averiguar.** ¿La lista personalizada supera a la lista de éxitos, o recomendar lo que ya es popular acierta igual de bien?

- **Qué decide:** si se paga la infraestructura de personalización. La popularidad es un baseline muy exigente y prácticamente **sin costo**: si el recomendador resulta equivalente a ella, el gasto no queda justificado con estas cifras y la decisión se lleva a un **A/B online**.
- **Antes de mirar el resultado:** se miran **precisión@10** (de las 10 propuestas, cuántas gustan) y **NDCG@10** (si lo bueno queda arriba). Toda brecha se mide contra su **ruido**: la propia SVD varía **≈ 0,006** en NDCG@10 solo al cambiar la semilla. Si su ventaja supera ese margen con claridad, personaliza efectivamente. Si la brecha se queda en torno a **0,001**, es un **empate dentro del ruido** y proclamar personalización sería sobreinterpretar los datos. Ambos, eso sí, deberían superar al azar con holgura.

🔎 **Qué hace el código.** La siguiente celda parte los datos **por fecha** (pasado →
futuro), entrena la SVD con el pasado y calcula **precisión@10** y **NDCG@10** para tres
modelos —**SVD**, **popularidad** y **azar**—; así se compara el recomendador
personalizado contra un baseline muy exigente (los éxitos) y contra el suelo.

In [ ]:
# Métricas top-k sencillas (precisión@k y NDCG@k, relevancia binaria: nota real >= 3.5)
def metricas_topk(preds, k=10, umbral=3.5):
    por_usuario = defaultdict(list)
    for uid, _, real, est, _ in preds:
        por_usuario[uid].append((est, real))
    precs, ndcgs = [], []
    for _, lst in por_usuario.items():
        lst.sort(key=lambda x: x[0], reverse=True)
        topk = lst[:k]
        rec = sum(e >= umbral for (e, _) in topk)
        aciertos = sum((e >= umbral) and (r >= umbral) for (e, r) in topk)
        precs.append(aciertos / rec if rec else 0)
        dcg = sum((1.0 if r >= umbral else 0.0) / np.log2(i + 2) for i, (_, r) in enumerate(topk))
        ideal = sorted((1.0 if r >= umbral else 0.0 for (_, r) in lst), reverse=True)[:k]
        idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
        ndcgs.append(dcg / idcg if idcg else 0)
    return float(np.mean(precs)), float(np.mean(ndcgs))

# Partición TEMPORAL: pasado -> futuro
orden = ratings.sort_values("timestamp")
corte = int(len(orden) * 0.8)
tr_df, te_df = orden.iloc[:corte], orden.iloc[corte:]
print(f"Partición temporal 80/20 por fecha:  train (pasado) = {len(tr_df):,}   test (futuro) = {len(te_df):,}")
tr = Dataset.load_from_df(tr_df[["user_id","item_id","rating"]], reader).build_full_trainset()
te = list(te_df[["user_id","item_id","rating"]].itertuples(index=False, name=None))

# SVD entrenado con el pasado
svd_t = SVD(n_factors=100, n_epochs=20, random_state=42).fit(tr)
p_svd, ndcg_svd = metricas_topk(svd_t.test(te), k=10)

# Baseline POPULARIDAD (nota media del ítem en el pasado)
media_item = {i: np.mean([r for (_, r) in tr.ir[i]]) for i in tr.all_items()}
def pop_pred(tuples):
    out = []
    for (u, i, r) in tuples:
        try: est = media_item.get(tr.to_inner_iid(i), tr.global_mean)
        except ValueError: est = tr.global_mean
        out.append((u, i, r, est, None))
    return out
p_pop, ndcg_pop = metricas_topk(pop_pred(te), k=10)

# Baseline AZAR (nota uniforme)
rng = np.random.RandomState(42)
p_azar, ndcg_azar = metricas_topk([(u, i, r, rng.uniform(1, 5), None) for (u, i, r) in te], k=10)

topk = pd.DataFrame([
    ("SVD (personalizado)", p_svd, ndcg_svd),
    ("Popularidad",         p_pop, ndcg_pop),
    ("Azar",                p_azar, ndcg_azar),
], columns=["modelo", "precisión@10", "NDCG@10"])
print(topk.round(3).to_string(index=False))

# ¿La ventaja de la SVD sobre la popularidad es señal o ruido?
# La SVD varía ~0,006 en NDCG@10 SOLO por cambiar la semilla (ruido de muestreo).
RUIDO_NDCG = 0.006          # variación de la SVD entre semillas (banda de ruido)
delta = abs(ndcg_svd - ndcg_pop)
print(f"\nNDCG@10:  SVD {ndcg_svd:.3f}  vs  popularidad {ndcg_pop:.3f}   ->   |Δ| = {delta:.4f}")
print(f"Ruido entre semillas ~ {RUIDO_NDCG:.3f}.  |Δ| = {delta:.4f} {'<' if delta < RUIDO_NDCG else '>='} ruido")
if delta < RUIDO_NDCG:
    print("=> EMPATE dentro del ruido: NO se afirma que la SVD personalice el orden; se decide con un A/B online.")
else:
    print("=> La brecha supera al ruido: hay indicio de personalización (confirmar igualmente con A/B).")

📖 **Lectura de negocio.** Sobre **partición temporal** (predecir el futuro con el pasado),
la **SVD** y la **popularidad** quedan **prácticamente iguales** en las dos métricas top-k:
precisión@10 ≈ 0,72 y NDCG@10 ≈ 0,78 en ambas. La brecha de NDCG@10 (**|Δ| ≈ 0,001**) es
**menor que la variación de la propia SVD entre semillas** (**≈ 0,006**): es un
**empate dentro del ruido**, **no** una prueba de que la SVD personalice el orden. La
**popularidad es un baseline muy exigente** (recomendar los éxitos acierta a menudo). Que un
recomendador **personalice** solo se afirma si su ventaja de ranking **supera con claridad al
ruido**; con estas cifras la decisión se lleva a un **A/B online**. Ambos, eso sí, superan con
claridad al **azar**. Conclusión: toda métrica top-k se lee **contra los baselines** (popularidad
y azar) **y contra su ruido**, nunca aislada.

💡 **Intuición (NDCG = ordenar el escaparate).** La precisión@10 pregunta «de los 10,
¿cuántos gustan?»; el **NDCG@10** premia además **poner lo mejor arriba**, donde caen los
clics. Acertar en el puesto 1 vale más que en el 10.

⚠️ **Alerta (la métrica determina el modelo superior… y se mide contra su ruido).** La **popularidad es un
baseline muy exigente**: recomendar los éxitos acierta a menudo y aquí **resulta equivalente** a la SVD tanto
en precisión@10 como en NDCG@10 (la SVD solo va por delante en RMSE). La brecha de NDCG@10 es
**más pequeña que el ruido entre semillas** (≈ 0,006), así que **no** demuestra personalización.
Regla de oro: elegir la métrica por el **objetivo de negocio**, compararla **siempre contra la
popularidad y el azar**, y **contrastar toda brecha de ranking contra su ruido**. Reportar NDCG
sin baseline de popularidad —o leer una brecha del tamaño del ruido como una ventaja real— induce a error.

---
## e) Recomendador de negocio: feedback implícito (Online Retail)

En el negocio real rara vez hay estrellas: solo **interacciones** (clics, **compras**,
tiempo). Es **feedback implícito**. La señal aquí son las **compras** de Online Retail II
(un e-commerce): cuánto compró cada cliente de cada producto. La idea (Hu, 2008): una
compra repetida da más **confianza** de que el producto gusta; **un cero es «no
observado», no «lo detesta»**. Se factoriza esa matriz de compras (`implicit` ALS) y se
obtiene el **top-N por cliente**.

**❓ Qué se quiere averiguar.** En un e-commerce no hay estrellas, solo compras. ¿Qué tres productos entran en el próximo email de cada cliente?

- **Qué decide:** la campaña de **cross-selling**: qué complemento aparece en la ficha, en el carrito o en el correo. Cruzado con el precio, ese top-N se vuelve ingreso potencial y señala dónde conviene reforzar la recomendación.
- **Antes de mirar el resultado:** aquí **no** habrá RMSE, porque no hay nota que predecir; lo que el modelo pondera es la **confianza**, y una compra repetida pesa más que una compra aislada. Conviene fijar la lectura antes de ver las listas: un **cero** en esa matriz significa **«no observado»**, no «no le gusta» —quizá el cliente nunca vio ese producto—. Si se leyera como rechazo, el recomendador se encerraría en lo ya conocido y su desempeño offline sería una **cota optimista**.

🔎 **Qué hace el código.** La siguiente celda arma la matriz **cliente × producto** con
las cantidades compradas (señal implícita), la factoriza con `implicit` **ALS**
ponderando por **confianza** (`c = 1 + 40·cantidad`) y devuelve el **top-3 por cliente**;
no se calcula RMSE porque **no hay nota** que predecir.

In [ ]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

retail = _cargar("online_retail_reco_muestra.csv")
print(f"Online Retail II (muestra): {len(retail):,} líneas | "
      f"{retail['Customer ID'].nunique()} clientes | {retail['StockCode'].nunique()} productos")

# Matriz cliente-producto con la CANTIDAD comprada agregada (señal implícita)
agg = retail.groupby(["Customer ID", "StockCode"])["Quantity"].sum().reset_index()
agg = agg[agg["Quantity"] > 0]
cust = agg["Customer ID"].astype("category"); prod = agg["StockCode"].astype("category")
ui = csr_matrix((agg["Quantity"].astype(float), (cust.cat.codes, prod.cat.codes)),
                shape=(len(cust.cat.categories), len(prod.cat.categories)))
dens_impl = ui.nnz / (ui.shape[0] * ui.shape[1])
print(f"Matriz cliente-producto: {ui.shape[0]} x {ui.shape[1]} | {ui.nnz:,} celdas no nulas "
      f"| densidad {dens_impl:.1%}")
print("No hay 'rating': la señal es la compra. No se calcula RMSE (no hay nota que predecir).")

# Confianza = 1 + 40*cantidad (Hu 2008); ALS factoriza ponderando por confianza.
# factors=32 (matriz pequeña), regularization=0.05 e iterations=15 son valores estándar
# de `implicit`; num_threads=1 fija el orden de reducción -> resultado REPRODUCIBLE entre máquinas.
als = AlternatingLeastSquares(factors=32, regularization=0.05, iterations=15,
                              random_state=42, num_threads=1)
als.fit((ui * 40).astype("float32"))

desc = retail.drop_duplicates("StockCode").set_index("StockCode")["Description"].to_dict()
precio = retail.groupby("StockCode")["Price"].mean().to_dict()
productos = prod.cat.categories; clientes = cust.cat.categories

filas = []
for f in range(min(5, len(clientes))):
    ids, sc = als.recommend(f, ui[f], N=3, filter_already_liked_items=True)
    for i, s in zip(ids, sc):
        p = productos[i]
        filas.append({"cliente": int(clientes[f]), "producto": p,
                      "descripción": str(desc.get(p, "?"))[:34],
                      "precio": round(float(precio.get(p, 0)), 2), "confianza": round(float(s), 3)})
reco_impl = pd.DataFrame(filas)
print("\nTop-3 recomendaciones por cliente (muestra de 5 clientes) con precio unitario:")
print(reco_impl.to_string(index=False))
print(f"\nTicket potencial medio de las recomendaciones: {reco_impl['precio'].mean():.2f} GBP/unidad.")

📖 **Lectura de negocio.** El recomendador implícito lee cada lista top-N como **próximas
compras probables** por cliente: es materia prima directa de **cross-selling**
(complementos en la ficha o el carrito), **up-selling** y **retención** (email
personalizado). La **confianza** pondera cuánto creer cada interacción (una compra
repetida pesa más que una vista aislada). Cruzar el top-N con el **precio** lo convierte
en **ingreso potencial** y señala dónde conviene reforzar la recomendación.

💡 **Intuición (confianza ≠ preferencia).** En implícito no se observa cuánto gusta algo,
solo cuánto se **interactuó**. La **confianza** traduce esa intensidad: una compra
repetida pesa más que una compra aislada, y ordena cuánto creer cada recomendación.

⚠️ **Alerta (MNAR: «no comprado» ≠ «no gusta»).** Un **cero** en la matriz de compras es
**«no observado»**, no «lo detesta»: quizá el cliente **nunca vio** ese producto. Tratar
el no-comprado como rechazo sesga el modelo hacia lo ya conocido y penaliza lo no
expuesto. Además, lo que la gente compra **no es una muestra al azar** del catálogo
(*missing-not-at-random*), así que el desempeño offline es una **cota optimista**.

---
## f) Exportación a Excel + figuras de resultados

Convención del curso: los resultados se exportan a `resultados/E07_resultados.xlsx` y
las **figuras de resultados se generan leyendo ese Excel** (no de objetos en memoria).

> **Nota de reproducibilidad.** Los valores operativos de E7 provienen de **su
> propia réplica** (un único split 80/20, semilla 42) y son **deterministas dentro del
> redondeo**: al re-ejecutar el cuaderno con la misma semilla se obtienen las mismas
> cifras (item-based ≈ **0,917**, media ≈ **1,130**, SVD ≈ **0,935**).

In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# Curva de cola larga (popularidad de las películas) para la figura de dispersión
por_item = ratings["item_id"].value_counts().sort_values(ascending=False).reset_index()
por_item.columns = ["item_id", "n_calif"]; por_item["rank"] = np.arange(1, len(por_item)+1)

wb = Workbook(); ws = wb.active; ws.title = "recsys"
ws["A1"] = "métrica"; ws["B1"] = "valor"
claves = [
    ("RMSE_SVD", rmse_svd), ("RMSE_baseline_media", rmse_media),
    ("RMSE_item_based", rmse_item), ("RMSE_user_based", rmse_user),
    ("NDCG10_SVD", ndcg_svd), ("NDCG10_popularidad", ndcg_pop),
    ("precision10_SVD", p_svd), ("precision10_azar", p_azar),
]
for r, (k, v) in enumerate(claves, start=2):
    ws[f"A{r}"] = k; ws[f"B{r}"] = float(v)
apoyo = [("n_calificaciones", n), ("n_usuarios", nu), ("n_items", ni),
         ("media_global", mu), ("densidad", densidad), ("RMSE_azar", rmse_azar),
         ("MAE_SVD", mae_svd), ("NDCG10_azar", ndcg_azar),
         ("precision10_popularidad", p_pop),
         ("implicit_clientes", int(ui.shape[0])), ("implicit_productos", int(ui.shape[1])),
         ("implicit_densidad", float(dens_impl))]
r0 = 12; ws[f"A{r0-1}"] = "APOYO"
for k, v in apoyo:
    ws[f"A{r0}"] = k; ws[f"B{r0}"] = v; r0 += 1

def add_df(name, df):
    w = wb.create_sheet(name)
    for row in dataframe_to_rows(df, index=False, header=True):
        w.append(row)
add_df("rmse_modelos", tabla_rmse.round(6))
add_df("topk", topk.round(6))
add_df("cola_larga", por_item[["rank", "item_id", "n_calif"]])
add_df("reco_implicit", reco_impl)
wb.save(XLSX)
print("Guardado:", XLSX)
print("Hojas:", wb.sheetnames)

In [ ]:
# Figuras de resultados — SE GENERAN LEYENDO EL EXCEL (convención del curso)
wbf = load_workbook(XLSX, data_only=True)
def leer(hoja):
    return (pd.DataFrame(wbf[hoja].values)
              .pipe(lambda d: d.rename(columns=d.iloc[0]).drop(0).reset_index(drop=True)))

# 1) RMSE por modelo
ra = leer("rmse_modelos"); ra["RMSE"] = ra["RMSE"].astype(float); ra = ra.sort_values("RMSE")
fig, ax = plt.subplots(figsize=(7.4, 4.0))
cols = [UPC_RED if "SVD" in m else (UPC_MAROON if "item" in m else GRIS) for m in ra["modelo"]]
b = ax.barh(ra["modelo"], ra["RMSE"], color=cols)
for bar, v in zip(b, ra["RMSE"]):
    ax.text(v + 0.02, bar.get_y()+bar.get_height()/2, f"{v:.2f}", va="center", fontsize=9)
ax.set_xlabel("RMSE (menor = mejor)"); ax.set_xlim(0, 1.75); ax.invert_yaxis()
ax.set_title("Error de predicción por modelo (MovieLens 100k)")
fig.savefig(os.path.join(FIG, "E07_rmse_modelos.png"), bbox_inches="tight"); plt.show()

# 2) precisión@10 y NDCG@10 por modelo
tk = leer("topk")
for c in ["precisión@10", "NDCG@10"]:
    tk[c] = tk[c].astype(float)
x = np.arange(len(tk)); w = 0.38
fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.bar(x - w/2, tk["NDCG@10"], w, color=UPC_RED, label="NDCG@10 (orden)")
ax.bar(x + w/2, tk["precisión@10"], w, color=GRIS, label="precisión@10")
for xi, (nd, pr) in enumerate(zip(tk["NDCG@10"], tk["precisión@10"])):
    ax.text(xi - w/2, nd + 0.01, f"{nd:.2f}", ha="center", fontsize=9)
    ax.text(xi + w/2, pr + 0.01, f"{pr:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(tk["modelo"]); ax.set_ylim(0, 1.0)
ax.set_ylabel("métrica top-k (mayor = mejor)")
ax.set_title("Evaluación top-k (partición temporal): SVD y popularidad empatan (dentro del ruido)")
ax.legend()
fig.savefig(os.path.join(FIG, "E07_topk_modelos.png"), bbox_inches="tight"); plt.show()

# 3) Cola larga (cold start de ítems)
dp = leer("cola_larga"); dp["rank"] = dp["rank"].astype(float); dp["n_calif"] = dp["n_calif"].astype(float)
fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.fill_between(dp["rank"], dp["n_calif"], color=UPC_RED, alpha=0.25)
ax.plot(dp["rank"], dp["n_calif"], color=UPC_RED, lw=1.8)
ax.set_xlabel("películas ordenadas por popularidad (rank)")
ax.set_ylabel("nº de calificaciones")
ax.set_title("Cola larga: pocos éxitos y muchísimos ítems con casi nada (cold start)")
fig.savefig(os.path.join(FIG, "E07_cola_larga.png"), bbox_inches="tight"); plt.show()
print("Figuras guardadas en:", FIG)

---
## g) Cierre del proyecto integrador — del análisis a la recomendación accionable

Esta es la **última sesión** del curso EPE. El proyecto integrador **integra todo lo
aprendido** en un caso de negocio real, recorrido de punta a punta con **CRISP-DM**:

| Fase CRISP-DM | Qué aporta el curso EPE |
|---|---|
| 1. Comprensión del negocio | Ficha CRISP-DM: problema, KPI, decisión (E1). |
| 2. Comprensión de los datos | EDA, estadística, A/B (E1). |
| 3. Preparación de los datos | Limpieza; reducción de dimensionalidad / PCA (E3). |
| 4. Modelado | Regresión (E2), regularización/PCA (E3), segmentación y reglas de asociación (E4), churn/fraude (E5), pronóstico y causalidad (E6), **recomendación (E7)**. |
| 5. Evaluación | Cada técnica con su métrica; se justifica la elegida. |
| 6. Comunicación | **Informe ejecutivo + sustentación** (el foco del cierre). |

**Cómo integra el recomendador todo el curso.** Se **segmenta** a los clientes (E4),
se prioriza a quién recomendar por su valor/riesgo de fuga (**churn**, E5), se
**pronostica** la demanda de lo recomendado (E6), y se **personaliza** con el
recomendador (E7). El resultado es una **acción sobre cada cliente**.

**La regla de oro de la comunicación ejecutiva: problema → evidencia → recomendación.**
El detalle técnico vive en el repositorio; ante la gerencia se presenta la decisión.

### Entregable de la sesión
Ver `evaluacion/entregable.docx`: un **recomendador con top-N + evaluación simple
(precisión@10 / NDCG@10)** y su lectura de negocio, más el **mensaje ejecutivo del
proyecto integrador**. Se apoya en `plantillas/guia_proyecto_integrador_epe.docx`
(guía + rúbrica), `plantillas/repositorio_plantilla_epe.docx` y
`plantillas/plantilla_sustentacion_epe.docx`. Rúbrica **vigesimal (total = 20)**. Al
cierre se aplica un **control corto**.

### Drills (enunciados; se resuelven en `evaluacion/drills.docx`)
1. **Construir un recomendador por similitud entre productos** (item-based): armar la
   matriz, medir la dispersión y explicar qué significa que dos productos sean «vecinos».
2. **Generar las 5 recomendaciones principales para un usuario** (colaborativo vs.
   contenido) y **discutirlas**: sobre-especialización, cold start y cuándo conviene cada uno.
3. **Preparar el mensaje ejecutivo del proyecto integrador** (problema → hallazgo →
   acción), legible por un directivo y trazable a las cifras del repositorio.

### Para seguir explorando (fuentes de actualidad)
Casos y estudios recientes sobre recomendación aplicada a negocio, con enlaces
verificados, en las fuentes de actualidad de la sesión. Ejemplos: Netflix cuantifica el
**valor económico** de recomendar (arXiv 2511.07280, 2025-26); Shopify explica
content-based / colaborativo / híbrido en e-commerce (03/04/2026).

### Bibliografía (lecturas de la sesión)
- Ricci, F., Rokach, L. & Shapira, B. (2015). *Recommender Systems Handbook*, 2.ª ed. Springer (capítulos introductorios).
- Provost, F. & Fawcett, T. (2013). *Data Science for Business*, cap. 12. O'Reilly.
- Koren, Y., Bell, R. & Volinsky, C. (2009). *Matrix Factorization Techniques for Recommender Systems*. IEEE Computer 42(8):30-37 (fondo intuitivo de la factorización).
- Hu, Y., Koren, Y. & Volinsky, C. (2008). *Collaborative Filtering for Implicit Feedback Datasets*. IEEE ICDM '08 (feedback implícito).

---
*Cuaderno de la Sesión EPE E7 — Herramientas de Ciencias de Datos — UPC. Sistemas de
recomendación y cierre del proyecto integrador. Datos: MovieLens 100k (GroupLens) y
Online Retail II (UCI 502). Todas las cifras provienen de la ejecución de este
cuaderno y de `resultados/E07_resultados.xlsx`. Deep learning de recomendación fuera de
alcance (solo mención).*